python 环境：3.14.3
必需第三方库：numpy, scikit-learn, tqdm

In [1]:
import numpy as np
import os
import glob
import random
import datetime
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, log_loss, f1_score

# ========== 配置 ==========
DATA_DIR = "./training_set"           # 包含所有 mv_*.txt 的文件夹
FACTOR_DIM = 80
LEARNING_RATE = 0.01
REG = 0.02
EPOCHS = 8
SEED = 42
VAL_SAMPLE_RATE = 0.001              # 采样率（0.1% ≈ 10万条）
RANDOM_SEED = 42

np.random.seed(SEED)
random.seed(RANDOM_SEED)

# ========== 辅助函数 ==========
def rating_to_label(rating):
    return 1 if rating >= 4 else 0

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# ========== 1. 建立用户/电影映射（同时统计训练集样本总数）==========
def build_mappings_from_files():
    user_set = set()
    movie_set = set()
    total_samples = 0                   # 用于统计训练集样本总数
    file_pattern = os.path.join(DATA_DIR, "mv_*.txt")
    file_list = glob.glob(file_pattern)
    print(f"找到 {len(file_list)} 个电影文件")
    for file_path in tqdm(file_list, desc="扫描文件建立映射"):
        basename = os.path.basename(file_path)
        movie_id = int(basename[3:-4])
        movie_set.add(movie_id)
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.endswith(':'):
                    continue
                parts = line.split(',')
                if len(parts) < 2:
                    continue
                user_id = int(parts[0])
                user_set.add(user_id)
                total_samples += 1       # 计数
    print(f"用户数: {len(user_set)}, 电影数: {len(movie_set)}")
    print(f"训练集总样本数: {total_samples}")
    user2idx = {uid: i for i, uid in enumerate(user_set)}
    movie2idx = {mid: i for i, mid in enumerate(movie_set)}
    return user2idx, movie2idx

user2idx, movie2idx = build_mappings_from_files()
num_users = len(user2idx)
num_movies = len(movie2idx)

# ========== 2. 从训练集中采样验证集 ==========
def build_validation_set(sample_rate=VAL_SAMPLE_RATE):
    val_samples = []   # 存储 (user_id, movie_id, label)
    file_pattern = os.path.join(DATA_DIR, "mv_*.txt")
    file_list = glob.glob(file_pattern)
    for file_path in tqdm(file_list, desc="构建验证集"):
        basename = os.path.basename(file_path)
        movie_id = int(basename[3:-4])
        with open(file_path, 'r') as f:
            lines = f.readlines()
        for line in lines:
            line = line.strip()
            if not line or line.endswith(':'):
                continue
            parts = line.split(',')
            if len(parts) < 2:
                continue
            if random.random() < sample_rate:
                user_id = int(parts[0])
                rating = int(parts[1])
                label = rating_to_label(rating)
                val_samples.append((user_id, movie_id, label))
    print(f"验证集样本数: {len(val_samples)}")
    return val_samples

val_samples = build_validation_set()

# ========== 3. 初始化参数 ==========
U = np.random.normal(0, 0.1, (num_users, FACTOR_DIM))
V = np.random.normal(0, 0.1, (num_movies, FACTOR_DIM))
user_bias = np.zeros(num_users)
movie_bias = np.zeros(num_movies)
global_bias = 0.0

# ========== 4. 训练一个 epoch ==========
def train_one_epoch(epoch):
    global U, V, user_bias, movie_bias, global_bias
    total_log_loss = 0.0
    num_samples = 0
    file_pattern = os.path.join(DATA_DIR, "mv_*.txt")
    file_list = glob.glob(file_pattern)
    for file_path in tqdm(file_list, desc=f"Epoch {epoch+1} - 训练", leave=False):
        basename = os.path.basename(file_path)
        movie_id = int(basename[3:-4])
        movie_idx = movie2idx[movie_id]
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.endswith(':'):
                    continue
                parts = line.split(',')
                if len(parts) < 2:
                    continue
                user_id = int(parts[0])
                rating = int(parts[1])
                label = rating_to_label(rating)
                user_idx = user2idx[user_id]
                
                # 前向传播
                raw = np.dot(U[user_idx], V[movie_idx]) + user_bias[user_idx] + movie_bias[movie_idx] + global_bias
                pred = sigmoid(raw)
                error = pred - label
                
                # 梯度
                grad_u = error * V[movie_idx]
                grad_v = error * U[user_idx]
                grad_bias_u = error
                grad_bias_v = error
                grad_global = error
                
                # 正则化
                grad_u += REG * U[user_idx]
                grad_v += REG * V[movie_idx]
                grad_bias_u += REG * user_bias[user_idx]
                grad_bias_v += REG * movie_bias[movie_idx]
                grad_global += REG * global_bias
                
                # 更新
                U[user_idx] -= LEARNING_RATE * grad_u
                V[movie_idx] -= LEARNING_RATE * grad_v
                user_bias[user_idx] -= LEARNING_RATE * grad_bias_u
                movie_bias[movie_idx] -= LEARNING_RATE * grad_bias_v
                global_bias -= LEARNING_RATE * grad_global
                
                # 累计损失
                total_log_loss += - (label * np.log(pred + 1e-15) + (1-label) * np.log(1-pred + 1e-15))
                num_samples += 1
    if num_samples == 0:
        print("错误：没有读取到任何训练样本")
        return float('inf')
    avg_log_loss = total_log_loss / num_samples
    print(f"Epoch {epoch+1} 训练平均 Log Loss: {avg_log_loss:.6f}")
    return avg_log_loss

# ========== 5. 在验证集上评估 ==========
def evaluate_validation():
    y_true = []
    y_pred = []
    for (user_id, movie_id, label) in val_samples:
        user_idx = user2idx.get(user_id)
        movie_idx = movie2idx.get(movie_id)
        if user_idx is None or movie_idx is None:
            pred = 0.5
        else:
            raw = np.dot(U[user_idx], V[movie_idx]) + user_bias[user_idx] + movie_bias[movie_idx] + global_bias
            pred = sigmoid(raw)
        y_true.append(label)
        y_pred.append(pred)
    auc = roc_auc_score(y_true, y_pred)
    logloss = log_loss(y_true, y_pred)
    y_pred_class = (np.array(y_pred) >= 0.5).astype(int)
    f1 = f1_score(y_true, y_pred_class)
    print(f"验证集结果: AUC = {auc:.5f}, Log Loss = {logloss:.5f}, F1 (0.5) = {f1:.5f}")
    return auc, logloss, f1

# ========== 6. 训练循环（记录最佳指标） ==========
best_auc = 0.0
best_logloss = None
best_f1 = None

for epoch in range(EPOCHS):
    train_one_epoch(epoch)
    auc, logloss, f1 = evaluate_validation()
    if auc > best_auc:
        best_auc = auc
        best_logloss = logloss
        best_f1 = f1
        np.savez(os.path.join(DATA_DIR, "best_model.npz"),
                 U=U, V=V, user_bias=user_bias, movie_bias=movie_bias, global_bias=global_bias)
        print("  -> 保存最佳模型")

# 输出最终评估标准
print("\n========== 最终模型评估标准（基于验证集） ==========")
print(f"最佳 AUC: {best_auc:.5f}")
print(f"对应 Log Loss: {best_logloss:.5f}")
print(f"对应 F1 Score (阈值0.5): {best_f1:.5f}")
print("================================================\n")

# ========== 7. 对 qualifying.txt 生成预测（带时间戳文件名） ==========
def predict_qualifying(output_file=None):
    if output_file is None:
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        output_file = f"predictions_{timestamp}.csv"
    
    QUAL_FILE = os.path.join(DATA_DIR, "qualifying.txt")
    if not os.path.exists(QUAL_FILE):
        QUAL_FILE = "qualifying.txt"
        if not os.path.exists(QUAL_FILE):
            print("qualifying.txt 不存在，无法预测")
            return
    with open(output_file, 'w') as out_f:
        out_f.write("Id,Probability,Label\n")
        with open(QUAL_FILE, 'r') as f:
            cur_movie = None
            for line in f:
                line = line.strip()
                if not line:
                    continue
                if line.endswith(':'):
                    cur_movie = int(line[:-1])
                    continue
                parts = line.split(',')
                if len(parts) < 1:
                    continue
                user_id = int(parts[0])
                user_idx = user2idx.get(user_id)
                movie_idx = movie2idx.get(cur_movie) if cur_movie is not None else None
                if user_idx is None or movie_idx is None:
                    prob = 0.5
                else:
                    raw = np.dot(U[user_idx], V[movie_idx]) + user_bias[user_idx] + movie_bias[movie_idx] + global_bias
                    prob = sigmoid(raw)
                label = 1 if prob >= 0.5 else 0
                out_f.write(f"{cur_movie}_{user_id},{prob:.6f},{label}\n")
    print(f"预测结果已保存到 {output_file}")

predict_qualifying()

找到 17770 个电影文件


扫描文件建立映射: 100%|██████████| 17770/17770 [00:28<00:00, 622.97it/s]


用户数: 480189, 电影数: 17770
训练集总样本数: 100480507


构建验证集: 100%|██████████| 17770/17770 [00:19<00:00, 892.33it/s] 


验证集样本数: 100351


...

Epoch 4 训练平均 Log Loss: 0.552594
验证集结果: AUC = 0.78567, Log Loss = 0.57038, F1 (0.5) = 0.73557
  -> 保存最佳模型


Epoch 5 训练平均 Log Loss: 0.545900
验证集结果: AUC = 0.79004, Log Loss = 0.56426, F1 (0.5) = 0.74113
  -> 保存最佳模型


Epoch 6 训练平均 Log Loss: 0.541258
验证集结果: AUC = 0.79370, Log Loss = 0.55972, F1 (0.5) = 0.74461
  -> 保存最佳模型


Epoch 7 训练平均 Log Loss: 0.537175
验证集结果: AUC = 0.79763, Log Loss = 0.55544, F1 (0.5) = 0.74746
  -> 保存最佳模型


Epoch 8 训练平均 Log Loss: 0.533183
验证集结果: AUC = 0.80163, Log Loss = 0.55127, F1 (0.5) = 0.75004
  -> 保存最佳模型

========== 最终模型评估标准（基于验证集） ==========
最佳 AUC: 0.80163
对应 Log Loss: 0.55127
对应 F1 Score (阈值0.5): 0.75004

预测结果已保存到 predictions_20260529_223031.csv
